# ENEC 2026 · Day 4 · Capstone: A-001 Evidence-Backed Review Briefing

**Time:** about 90 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Combine structured data, the forecast, enterprise documents and an agent into one governed briefing for a human reviewer. Then check it with transparent automatic checks and the course rubric.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## Scenario

An engineer has observed unusual behaviour around synthetic asset **A-001**.

> **What does the available evidence say about A-001, what should a qualified reviewer pay attention to, and what is the appropriate human next step?**

## Your workflow

1. Understand the business question
2. Inspect the structured data
3. Review the sensor trend / ML output
4. Retrieve enterprise documents
5. Confirm which document is authoritative
6. Combine evidence **without blurring the source types**
7. Identify conflicts, limits and missing evidence
8. Write a bounded briefing
9. **Stop at the human-review boundary**

The final result is a **decision-support briefing**, not an equipment diagnosis or maintenance authorisation.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

tables = enec.load_tables()
con    = enec.make_db(tables)
docs   = enec.load_documents(include_day4=True)         # 9 core documents + 7 Day 4 project documents
index  = enec.RAGIndex.build(client, docs)
hourly = tables["a001_sensor_hourly_90d"]
print(len(docs), "documents,", len(index.chunks), "chunks")

## Step 1. Which documents are authoritative?

Before reading any content, read the **register**. The capstone corpus includes a DRAFT procedure, a SUPERSEDED procedure, an OPEN work order and a DRAFT cross-functional risk register.

In [ ]:
register = pd.DataFrame([{k: d[k] for k in ("doc_id", "title", "status", "version", "date")} for d in docs])
register.sort_values(["status", "doc_id"])

**Write down:** which documents can you cite as *current authority*? Which must you flag? (Only APPROVED are authority. DRAFT, SUPERSEDED and OPEN need a caveat.)

## Step 2. Structured evidence (deterministic code, no LLM)

In [ ]:
enec.run_sql(con, "SELECT * FROM asset_360 WHERE asset_id='A-001'").T

In [ ]:
enec.run_sql(con, """SELECT work_order_id, work_type, priority, status, created_date, due_date, problem_description
FROM work_orders WHERE asset_id='A-001' AND status IN ('OPEN','IN_PROGRESS','DEFERRED')
ORDER BY created_date DESC""")

In [ ]:
enec.run_sql(con, """SELECT inspection_id, inspection_date, inspection_type, severity, finding, recommendation, follow_up_required
FROM inspections WHERE asset_id='A-001' ORDER BY inspection_date DESC LIMIT 5""")

In [ ]:
enec.run_sql(con, """SELECT project_id, risk_id, risk_category, exposure_score, risk_status, evidence_basis
FROM a001_project_risk_360 ORDER BY exposure_score DESC LIMIT 6""")

## Step 3. Predictive evidence: sensor trend and forecast

In [ ]:
by_month = enec.run_sql(con, """SELECT substr(hour_timestamp,1,7) AS month, ROUND(AVG(avg_vibration_mm_s),2) AS avg_vibration,
       ROUND(MAX(max_vibration_mm_s),2) AS max_vibration, ROUND(AVG(avg_temperature_c),1) AS avg_temp_c,
       SUM(anomaly_minutes) AS anomaly_minutes FROM a001_sensor_hourly_90d GROUP BY 1 ORDER BY 1""")
display(by_month)

fc = enec.forecast_vibration(hourly); future = fc.pop("_forecast_df")
print(json.dumps(fc, indent=2))

plt.figure(figsize=(13, 4))
tail = hourly.tail(24 * 14)
plt.plot(tail.hour_timestamp, tail.avg_vibration_mm_s, label="Last 14 days")
plt.plot(future.hour_timestamp, future.avg_vibration_mm_s, label="Forecast", color="#f5a623")
plt.legend(); plt.grid(alpha=.25); plt.title("A-001 vibration (mm/s)"); plt.show()

**What the forecast does not prove:** It extrapolates a pattern in a synthetic series. It does not identify a cause, a failure mode or a time to failure.

## Step 4. Document evidence

Retrieve from several angles so the briefing does not depend on one lucky search.

In [ ]:
seen, hits = set(), []
for q in enec.DOC_QUERIES:
    for h in index.search(q, k=3):
        key = (h["doc_id"], h["chunk_no"])
        if key not in seen:
            seen.add(key); hits.append(h)
pd.DataFrame([{"doc_id": h["doc_id"], "status": h["status"], "version": h["version"], "score": round(h["score"], 3),
               "excerpt": h["text"][:90].replace("\n", " ")} for h in hits]).sort_values("doc_id")

## Step 5. Assemble the evidence packet

The packet keeps the three evidence types **separate**. Keeping them separate is how you avoid blurring a database fact, a model output and a document claim into one confident sentence.

In [ ]:
packet = enec.build_evidence_packet(con, index, hourly, docs)
print({k: (len(v) if hasattr(v, "__len__") else v) for k, v in packet.items()})
text = enec.packet_to_text(packet)
print("packet size:", len(text), "characters")
print(text[:1500])

## Step 6. Generate the briefing

The model writes only from the packet, in the eight required sections.

In [ ]:
briefing = enec.generate_briefing(client, packet)
from IPython.display import Markdown
Markdown(briefing)

## Step 7. Check the briefing (transparent, rule-based)

These checks are deliberately simple so you can read and extend them. They support the human reviewer. They do not replace review.

In [ ]:
checks = enec.check_briefing(briefing, docs)
checks

### Optional: second opinion from an LLM judge using the course rubric (1-5 per dimension, max 25)

In [ ]:
verdict = enec.judge_briefing(client, briefing)
print(json.dumps(verdict, indent=2))
scores = verdict.get("scores", {})
if scores:
    print("Total:", sum(int(v) for v in scores.values()), "/ 25   (21-25 strong, 16-20 good, 11-15 partial, 5-10 needs remediation)")

## Step 8. Your review (the human part)

Read your briefing as the qualified reviewer would. Fill in this table by hand. **The score you give matters more than the judge's.**

| Dimension | What to look for | Your score (1-5) |
|---|---|---|
| Problem understanding | Is the business question framed, not just answered technically? | |
| Data use | Are observations separated from interpretation? | |
| AI / RAG use | Right documents retrieved? Status and version respected? | |
| Evidence & explainability | Can you trace each claim to a table or document ID? | |
| Governance & human oversight | Is uncertainty kept? Is the final decision left to a human? | |

In [ ]:
# Save the briefing so you can attach it to your capstone submission
path = "/content/A001_review_briefing.md" if enec.os.path.isdir("/content") else "A001_review_briefing.md"
open(path, "w").write(briefing)
print("saved:", path)
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## Capstone tasks

1. **Break it on purpose.** Re-run Step 6 after removing the `Authority rules` line from `enec.BRIEFING_SYSTEM`. Which checks fail?
2. **Find an evidence gap.** Which question can the packet not answer (for example: cost of downtime, spare-parts lead time)? Add a retrieval query or table that might close the gap, or state it explicitly in section 6.
3. **Conflict hunt.** Compare the DRAFT cross-functional risk register with an APPROVED document. Do they disagree? Does your briefing say which one governs?
4. **Present** your briefing in 3 minutes: the question, the evidence, the limits, and the human next step.